# Image Autoencoder with MNIST

This notebook trains **two autoencoders** on handwritten digit images (28×28 pixels):
1. A standard autoencoder for compression/reconstruction
2. A denoising autoencoder that learns to remove Gaussian noise

### Pipeline

1. **Data Loading** — Load MNIST (60K train / 10K test images) and normalize pixel values to [0, 1]
2. **Standard Autoencoder** — Encoder compresses 784 pixels → 25-dimensional bottleneck; decoder reconstructs back to 784
3. **Evaluation** — Compare original vs reconstructed test images side by side
4. **Denoising Autoencoder** — Same architecture but with `GaussianNoise(0.2)` injected after flattening; trained to recover clean images from noisy inputs
5. **Denoising Evaluation** — Feed noisy test images and verify the model removes the corruption

### Key Concepts

* **Deep bottleneck** — progressive compression (784→400→200→100→50→25) forces the network to learn a compact 25D representation
* **Sigmoid output** — constrains reconstructed pixels to [0, 1] matching the normalized input range
* **Binary crossentropy** — treats each pixel as an independent Bernoulli variable; works well for grayscale images in [0, 1]
* **GaussianNoise layer** — adds random noise during training only (inactive at inference), teaching the model to denoise
* **No GPU required** — MNIST is small enough for CPU training (~2-3 min per model)

In [0]:
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist

In [0]:
# Load MNIST: 60K training + 10K test images of handwritten digits (28x28 grayscale)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

# Normalize pixel values from [0, 255] to [0, 1] for stable training
X_train = X_train / 255.0
X_test = X_test / 255.0

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Reshape
from tensorflow.keras.optimizers import SGD

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [0]:
# Encoder: progressively compresses 784 pixels down to a 25-dimensional bottleneck
encoder = Sequential()
encoder.add(Flatten(input_shape=[28, 28]))  # 28x28 -> 784
encoder.add(Dense(400, activation='relu'))   # 784 -> 400
encoder.add(Dense(200, activation='relu'))   # 400 -> 200
encoder.add(Dense(100, activation='relu'))   # 200 -> 100
encoder.add(Dense(50, activation='relu'))    # 100 -> 50
encoder.add(Dense(25, activation='relu'))    # 50 -> 25 (bottleneck)

# Decoder: reconstructs the image from the 25D compressed representation
decoder = Sequential()
decoder.add(Dense(50, activation='relu', input_shape=[25]))   # 25 -> 50
decoder.add(Dense(100, activation='relu'))   # 50 -> 100
decoder.add(Dense(200, activation='relu'))   # 100 -> 200
decoder.add(Dense(400, activation='relu'))   # 200 -> 400
decoder.add(Dense(784, activation='sigmoid'))  # 400 -> 784 (sigmoid constrains to [0,1])
decoder.add(Reshape([28, 28]))               # reshape flat 784 back to 28x28 image

# Combine encoder + decoder into a single autoencoder model
autoencoder = Sequential([encoder, decoder])
autoencoder.compile(optimizer=SGD(learning_rate=1.5), loss='binary_crossentropy', metrics=['accuracy'])

# Train: input = output (learn to reconstruct the original images through the bottleneck)
autoencoder.fit(X_train, X_train, epochs=5, validation_data=[X_test, X_test])

In [0]:
# Pass 10 test images through the full autoencoder (encode -> decode)
passed_images = autoencoder.predict(X_test[:10])

# Display originals (top row) vs reconstructions (bottom row)
fig, axes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    # Top row: original test images
    axes[0, i].imshow(X_test[i], cmap='gray')
    axes[0, i].axis('off')
    # Bottom row: autoencoder reconstructions
    axes[1, i].imshow(passed_images[i], cmap='gray')
    axes[1, i].axis('off')

axes[0, 0].set_title('Originals', fontsize=12, loc='left')
axes[1, 0].set_title('Reconstructed', fontsize=12, loc='left')
plt.tight_layout()
plt.show()

---
## Part 2: Denoising Autoencoder

The standard autoencoder above learns a compressed representation. Now we extend it to **denoise** corrupted images by injecting `GaussianNoise(0.2)` into the encoder during training.

The model learns to map noisy inputs back to clean targets — effectively learning which patterns are signal vs noise. At test time, we manually corrupt images and verify the network removes the corruption.

In [0]:
from tensorflow.keras.layers import GaussianNoise
import tensorflow as tf

# Set seeds for reproducibility
tf.random.set_seed(101)
np.random.seed(101)

# --- Denoising Autoencoder ---
# Same architecture as above, but with GaussianNoise injected after flattening.
# During training, the noise layer corrupts the input; the network learns to
# reconstruct the CLEAN image despite the corruption. At inference time,
# GaussianNoise is inactive — we add noise manually to test denoising.

# Encoder with noise injection
encoder = Sequential()
encoder.add(Flatten(input_shape=[28, 28]))    # 28x28 -> 784
encoder.add(GaussianNoise(0.2))               # inject Gaussian noise (std=0.2) during training
encoder.add(Dense(400, activation='relu'))     # 784 -> 400
encoder.add(Dense(200, activation='relu'))     # 400 -> 200
encoder.add(Dense(100, activation='relu'))     # 200 -> 100
encoder.add(Dense(50, activation='relu'))      # 100 -> 50
encoder.add(Dense(25, activation='relu'))      # 50 -> 25 (bottleneck)

# Decoder: same as standard autoencoder
decoder = Sequential()
decoder.add(Dense(50, activation='relu', input_shape=[25]))   # 25 -> 50
decoder.add(Dense(100, activation='relu'))     # 50 -> 100
decoder.add(Dense(200, activation='relu'))     # 100 -> 200
decoder.add(Dense(400, activation='relu'))     # 200 -> 400
decoder.add(Dense(784, activation='sigmoid'))  # 400 -> 784 (sigmoid: [0,1])
decoder.add(Reshape([28, 28]))                # reshape back to 28x28

# Combine into denoising autoencoder (Adam optimizer for smoother convergence)
noise_remover = Sequential([encoder, decoder])
noise_remover.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train: noisy input (via GaussianNoise layer) -> clean output target
noise_remover.fit(X_train, X_train, epochs=8)

In [0]:
# Create a noise-adding layer to manually corrupt test images
# (GaussianNoise is only active during training=True)
sample = GaussianNoise(0.2)
ten_noisy_images = sample(X_test[:10], training=True)

# Pass noisy images through the denoising autoencoder
denoised = noise_remover.predict(ten_noisy_images)

# Display: originals (top) vs denoised reconstructions (bottom)
fig, axes = plt.subplots(3, 10, figsize=(15, 4.5))
for i in range(10):
    # Top row: clean originals
    axes[0, i].imshow(X_test[i], cmap='gray')
    axes[0, i].axis('off')
    # Middle row: noisy inputs
    axes[1, i].imshow(ten_noisy_images[i], cmap='gray')
    axes[1, i].axis('off')
    # Bottom row: denoised outputs
    axes[2, i].imshow(denoised[i], cmap='gray')
    axes[2, i].axis('off')

axes[0, 0].set_title('Original', fontsize=11, loc='left')
axes[1, 0].set_title('Noisy Input', fontsize=11, loc='left')
axes[2, 0].set_title('Denoised', fontsize=11, loc='left')
plt.tight_layout()
plt.show()